# Feature Extraction with Transfer Learning

Feature Extraction is the first major stage of Transfer Learning.

In Feature Extraction, we use a pretrained CNN as a feature extractor.

The pretrained CNN is frozen, meaning its learned weights are not updated.

We then add a new classifier for our own problem and train only that classifier.

## What is Feature Extraction?

A pretrained CNN has already learned many useful visual features.

For example:

Image
 ↓
Edges
 ↓
Textures
 ↓
Shapes
 ↓
Object parts
 ↓
High-level features

Instead of training these features from scratch, we reuse them.

The pretrained CNN extracts useful features from our images.

Then a new classifier uses those features to make predictions.

## Feature Extraction Architecture

Our model will look like this:

                Input Image
                     ↓
              Pretrained CNN
              MobileNetV2
                     ↓
              Feature Maps
                     ↓
       Global Average Pooling
                     ↓
              Dense Layer
                     ↓
              Output Layer
                     ↓
                Cat / Dog

During Feature Extraction:

Pretrained CNN → FROZEN

New classifier → TRAINABLE

## What Does Frozen Mean?

When we freeze the pretrained CNN, its weights cannot be updated during training.

For example:

CNN Layer 1 → Frozen
CNN Layer 2 → Frozen
CNN Layer 3 → Frozen
CNN Layer 4 → Frozen
...
CNN Layer N → Frozen

New Dense Layer → Trainable

Therefore, during training:

Pretrained CNN weights → Don't change

New classifier weights → Change

## Why Do We Freeze the Pretrained CNN?

The pretrained CNN has already learned useful features from a huge dataset.

We don't want to destroy this knowledge immediately.

Freezing gives us several advantages:

- Faster training
- Less computation
- Fewer trainable parameters
- Lower risk of overfitting
- Useful when our dataset is small

After feature extraction, we can optionally fine-tune some CNN layers.

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## Step 1 — Load the Pretrained CNN

We will use MobileNetV2.

We set:

weights="imagenet"

to load the pretrained ImageNet weights.

We set:

include_top=False

because we don't want MobileNetV2's original ImageNet classifier.

We will create our own classifier for Cat vs Dog.

In [2]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

print("MobileNetV2 loaded!")

MobileNetV2 loaded!


## Step 2 — Freeze the Pretrained CNN

Now we freeze MobileNetV2.

This means its pretrained weights will not be updated during training.

In [3]:
base_model.trainable = False

print("Base model trainable:", base_model.trainable)

Base model trainable: False


## Step 3 — Add Our Own Classifier

MobileNetV2 was originally trained to classify ImageNet classes.

We don't need that classifier.

Instead, we will add:

MobileNetV2
↓
GlobalAveragePooling2D
↓
Dropout
↓
Dense
↓
Sigmoid
↓
Cat / Dog

The final Dense layer has one neuron because this is binary classification.

In [4]:
inputs = keras.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

x = layers.Dense(128, activation="relu")(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Understanding the Data Flow

Suppose we give the model an image:

Cat image
   ↓
224 × 224 × 3
   ↓
MobileNetV2
   ↓
Feature Maps
   ↓
Global Average Pooling
   ↓
Feature Vector
   ↓
Dense Layer
   ↓
Sigmoid
   ↓
0.95

If our threshold is 0.5:

0.95 > 0.5

Prediction = Dog/Cat depending on our class encoding.

## Global Average Pooling

The CNN produces feature maps.

For example:

Feature Maps
     ↓
Many spatial values
     ↓
GlobalAveragePooling2D
     ↓
One value per feature map

This converts the CNN's feature maps into a smaller feature vector.

It also greatly reduces the number of parameters compared with adding a large fully connected layer directly.